# Notebook 04: Monte Carlo and Thermalization

**Learning objectives:**
- Implement the Metropolis algorithm for lattice gauge theory
- Observe thermalization: how the plaquette evolves from cold/hot starts
- Understand the role of $\beta$ (coupling constant)
- Perform a quick plaquette-vs-$\beta$ scan

**Prerequisites:** Notebooks 01-03

**Pattern from:** `pvb.py` (Metropolis loop)

In [2]:
from notebook_utils import setup_paths, quick_metropolis
setup_paths()

import numpy as np
import matplotlib.pyplot as plt
import su2

## 1. The Metropolis Algorithm

We want to sample gauge configurations $\{U\}$ from the Boltzmann distribution
$$P[U] \propto e^{-S_G[U]}, \qquad S_G = \frac{\beta}{2}\sum_P \left(1 - \frac{1}{2}\mathrm{Re}\,\mathrm{Tr}\, U_P\right)$$

The Metropolis update for a single link $U_\mu(x)$:
1. Propose $U' = g \cdot U$ where $g \approx \mathbb{1}$ is a small random SU(2) matrix
2. Compute the change in action: $\Delta S = -\frac{\beta}{2}\,\mathrm{Re}\,\mathrm{Tr}\bigl[(U' - U) \cdot \Sigma_{\text{staples}}\bigr]$
3. Accept $U'$ if $\Delta S < 0$, or with probability $e^{-\Delta S}$ otherwise

This satisfies **detailed balance** and drives the system to equilibrium.

In [ ]:
La = [4, 4, 4, 4]
V = su2.vol(La)
beta = 2.4
Mlink = 10  # multi-hit: attempts per link

# Initialize cold start
U = np.zeros((V, 4, 4))
mups = np.zeros((V, 4), dtype=int)
mdns = np.zeros((V, 4), dtype=int)
for i in range(V):
    for mu in range(4):
        U[i][mu] = su2.cstart()
        mups[i, mu] = su2.mupi(i, mu, La)
        mdns[i, mu] = su2.mdowni(i, mu, La)

print(f"Initial plaquette: {su2.calcPlaq(U, La, mups):.6f}")

# === One sweep ===
accepted = 0
total = 0
for i in range(V):
    for mu in range(4):
        U0 = U[i][mu].copy()
        staples = su2.getstaple(U, i, mups, mdns, mu)
        for _ in range(Mlink):
            U0n = su2.update(U0)
            dS = -0.5 * su2.tr(su2.mult(U0n - U0, staples))
            if dS < 0 or np.random.random() < np.exp(-beta * dS):
                U[i][mu] = U0n
                U0 = U0n
                accepted += 1
            total += 1

print(f"After 1 sweep: plaquette = {su2.calcPlaq(U, La, mups):.6f}")
print(f"Acceptance rate: {accepted/total:.1%}")

## 2. Thermalization

Starting from a cold or hot configuration, the Monte Carlo evolves
the system toward equilibrium. The **thermalization** period must be
discarded before taking measurements.

Let's compare cold and hot starts.

In [ ]:
np.random.seed(42)
n_sweeps = 300
beta = 2.4

plaqs_cold, _ = quick_metropolis(La, beta=beta, n_sweeps=n_sweeps,
                                 start='cold', seed=42)
plaqs_hot, _  = quick_metropolis(La, beta=beta, n_sweeps=n_sweeps,
                                 start='hot', seed=43)

plt.figure(figsize=(8, 4))
plt.plot(plaqs_cold, label='Cold start', linewidth=1.5)
plt.plot(plaqs_hot,  label='Hot start', linewidth=1.5)
plt.xlabel('Sweep')
plt.ylabel(r'$\langle P \rangle$')
plt.title(f'Thermalization on $4^4$ lattice, $\\beta = {beta}$')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# After thermalization, both should converge
print(f"Cold (last 20 sweeps): {np.mean(plaqs_cold[-20:]):.4f}")
print(f"Hot  (last 20 sweeps): {np.mean(plaqs_hot[-20:]):.4f}")

Both starts converge to the same equilibrium value -- this is a key
consistency check for the Monte Carlo algorithm.

**Why don't they match exactly?** Even after thermalization, the plaquette
fluctuates from sweep to sweep. These are **statistical fluctuations**
inherent to Monte Carlo sampling: each sweep is a random walk in
configuration space, so $\langle P \rangle$ at any given sweep is a noisy
estimate of the true expectation value.

On a small $4^4$ lattice, the fluctuations are relatively large
(a few percent). Agreement within $\sim 5\%$ after sufficient sweeps
indicates convergence. To reduce the noise:
- **More sweeps**: averaging over more measurements beats down $1/\sqrt{N}$ errors
- **Larger volume**: more links per sweep $\Rightarrow$ smaller fluctuations per measurement
- **Ensemble averaging**: average over many independent configurations (this is what we do in production)

The hot start takes longer to thermalize than the cold start because it
begins far from equilibrium (random SU(2) matrices have $\langle P \rangle \approx 0$)
and must "cool down" to the equilibrium value, whereas the cold start
($P = 1$) only needs to "warm up" slightly.

## 3. Plaquette vs $\beta$ Scan

The coupling $\beta = 4/g^2$ controls the gauge dynamics:
- $\beta \to 0$ (strong coupling): $\langle P \rangle \to 0$
- $\beta \to \infty$ (weak coupling): $\langle P \rangle \to 1$

Analytical predictions:
- Strong coupling: $\langle P \rangle \approx \beta/4$ for $\beta \ll 1$
- Weak coupling: $\langle P \rangle \approx 1 - 3/(4\beta)$ for $\beta \gg 1$

In [ ]:
betas = [1.0, 2.0, 2.4, 3.0, 4.0]
plaq_means = []
plaq_errs = []

for b in betas:
    plaqs, _ = quick_metropolis(La, beta=b, n_sweeps=40,
                                start='cold', seed=100)
    # Discard first 15 sweeps (thermalization)
    meas = plaqs[15:]
    plaq_means.append(np.mean(meas))
    plaq_errs.append(np.std(meas) / np.sqrt(len(meas)))
    print(f"  beta={b:.1f}: P = {plaq_means[-1]:.4f} +/- {plaq_errs[-1]:.4f}")

In [ ]:
# Plot with analytical predictions
betas_arr = np.array(betas)
b_fine = np.linspace(0.5, 5, 100)
strong_coupling = b_fine / 4
weak_coupling = 1 - 3 / (4 * b_fine)

plt.figure(figsize=(8, 5))
plt.errorbar(betas, plaq_means, yerr=plaq_errs, fmt='ko',
             capsize=4, markersize=6, label='Monte Carlo')
plt.plot(b_fine, strong_coupling, 'b--', alpha=0.5,
         label=r'Strong coupling: $\beta/4$')
plt.plot(b_fine, weak_coupling, 'r--', alpha=0.5,
         label=r'Weak coupling: $1 - 3/(4\beta)$')
plt.xlabel(r'$\beta$', fontsize=13)
plt.ylabel(r'$\langle P \rangle$', fontsize=13)
plt.title(r'Plaquette vs $\beta$ on $4^4$ lattice')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercises

1. Run 200 sweeps and identify when thermalization ends by eye.
   How would you automate this detection?
2. Change `Mlink` from 10 to 1. How does the acceptance rate change?
   How does thermalization speed change? Fewer hits per link means
   lower acceptance but faster sweeps; too few hits means poor decorrelation.
3. **Coupling strength**: Change $\beta$ from 2.4 to 1.0 (strong coupling)
   or 4.0 (weak coupling) and re-run the thermalization comparison.
   Higher $\beta$ means weaker coupling, smoother fields, and higher
   acceptance. How does the equilibrium plaquette value change?
4. At $\beta = 2.4$, what is the expected plaquette from the weak-coupling
   formula? Compare to your measurement.
5. **Extended $\beta$ scan**: Add more $\beta$ values (e.g., 0.5, 5.0) to
   the scan in Section 3 to extend the range. How well do the strong-
   and weak-coupling predictions match at the extremes?
6. **Thermalization length**: Increase `n_sweeps` to 500 in Section 2. Does the
   hot start fully converge to the cold start's equilibrium value? How
   many sweeps does it take?
7. **Acceptance rate vs $\beta$**: Run short simulations (40 sweeps each)
   at $\beta = 0.5, 1.0, 2.0, 3.0, 4.0, 6.0$. Track the acceptance rate
   (you can modify `quick_metropolis` or write a version that returns it).
   Plot acceptance rate vs $\beta$. Why does acceptance increase with $\beta$?
   (*Hint:* at large $\beta$, the gauge field is already smooth, so small
   proposed changes are easily accepted.)
8. **Autocorrelation**: Using the plaquette time series from a 300-sweep run,
   compute the autocorrelation function
   $\rho(\Delta) = \frac{\langle P(n)P(n+\Delta)\rangle - \langle P\rangle^2}
   {\langle P^2\rangle - \langle P\rangle^2}$.
   After how many sweeps does $\rho$ drop below 0.1?
   This is roughly the decorrelation time.
   (*Use `from notebook_utils import autocorrelation` for convenience.*)